# Lung Cancer Classification using MobileNetV2

This notebook performs classification of lung histopathology images using both classical machine learning models and a deep learning model (MobileNetV2).

The workflow includes:
- Data acquisition and exploration
- Data preprocessing
- Baseline model comparison
- Advanced deep learning model
- Evaluation and visualization

How to run:
- Run all cells from top to bottom
- Ensure dataset is downloaded via Kaggle API

# Data Acquisition

In [ ]:
!pip install kaggle

In [ ]:
import os

# Set Kaggle API credentials manually before running

In [ ]:
!kaggle datasets download -d andrewmvd/lung-and-colon-cancer-histopathological-images

In [ ]:
!unzip lung-and-colon-cancer-histopathological-images.zip

In [ ]:
import random

dataset_path = "lung_colon_image_set/lung_image_sets"

image_paths = []
labels = []

for label in os.listdir(dataset_path):
    folder = os.path.join(dataset_path, label)

    for file in os.listdir(folder):
        image_paths.append(os.path.join(folder, file))
        labels.append(label)

print("Total images:", len(image_paths))

# EDA

In [ ]:
import cv2
import matplotlib.pyplot as plt

samples = random.sample(list(zip(image_paths, labels)), 20)

plt.figure(figsize=(10,10))

for i,(path,label) in enumerate(samples):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(4,5,i+1)
    plt.imshow(img)
    plt.title(label)
    plt.axis("off")

plt.show()

In [ ]:
sizes = []

for path in image_paths[:500]:
    img = cv2.imread(path)
    h,w,_ = img.shape
    sizes.append((h,w))

print("Min size:", min(sizes))
print("Max size:", max(sizes))

In [ ]:
formats = set()

for path in image_paths:
    formats.add(path.split('.')[-1])

print("Formats:", formats)

**EDA & Visual Inspection**

Setelah melakukan EDA menggunakan Google Colab, kami menemukan beberapa
key points. Kami menemukan bahwa size dari masing-masing gambar serta format gambar menampilkan hasil yang konsisten. Maximum dan minimum size data menunjukkan angka 768 x 768, sedangkan keseluruhan gambar berformat jpeg.

# Statistics & Data Quality

In [ ]:
corrupted = []
valid_images = []

for path in image_paths:
    try:
        img = cv2.imread(path)
        if img is None:
            corrupted.append(path)
        else:
            valid_images.append(path)
    except:
        corrupted.append(path)

print("Total images:", len(image_paths))
print("Valid images:", len(valid_images))
print("Corrupted images:", len(corrupted))

In [ ]:
import numpy as np

means = []
stds = []

for path in valid_images[:1000]:
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0

    means.append(img.mean(axis=(0,1)))
    stds.append(img.std(axis=(0,1)))

mean_rgb = np.mean(means, axis=0)
std_rgb = np.mean(stds, axis=0)

print("Mean RGB:", mean_rgb)
print("Std RGB:", std_rgb)

In [ ]:
from collections import Counter

class_counts = Counter(labels)

print(class_counts)

In [ ]:
classes = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(6,4))
plt.bar(classes, counts)

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")

plt.show()

**Statistics & Data Quality**

*   Dataset yang terdiri dari 15000 data image memiliki 0 corrupted/unreadable image, sehingga dapat disimpulkan bahwa semua data bernilai valid.
*   Dataset terdiri dari 3 jenis data, yakni gambar normal lung, lung adenocarcinoma, dan lung squamous cell carcinoma dengan masing-masing jenis berjumlah 5000.
*   Box plot menujukkan ukuran batang yang seragam karena data memiliki banyak nilai yang sama.
*   Size memiliki ukuran yang konsisten, sehingga dapat memudahkan langkah preprocessing.



# Data Preprocessing Pipeline

In [ ]:
import tensorflow as tf

data_dir = "lung_colon_image_set/lung_image_sets"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

# normalization
train_ds = train_ds.map(lambda x, y: (x/255.0, y))
val_ds = val_ds.map(lambda x, y: (x/255.0, y))

# optimization
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# check
for images, labels in train_ds.take(1):
    print(images.shape, labels.shape)

# Baseline


In [ ]:
import numpy as np
import cv2, os, random
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

dataset_path = "lung_colon_image_set/lung_image_sets"
IMG_SIZE     = (64, 64)
SAMPLE_PER_CLASS = 1000

In [ ]:
# Feature Extraction
def extract_hog(path, size=IMG_SIZE):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, size)
    return hog(img, orientations=9, pixels_per_cell=(8,8),
               cells_per_block=(2,2), visualize=False)

def extract_color_hist(path, size=IMG_SIZE, bins=32):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, size)
    feats = []
    for ch in range(3):
        h = cv2.calcHist([img], [ch], None, [bins], [0,256])
        feats.extend(h.flatten())
    return np.array(feats)

def extract_features(path):
    return np.concatenate([extract_hog(path), extract_color_hist(path)])


In [ ]:
# Load Data + Stratified Random Sampling
print("Loading data dengan Stratified Random Sampling...")

X_raw, y_raw = [], []

for label in sorted(os.listdir(dataset_path)):
    folder = os.path.join(dataset_path, label)
    if not os.path.isdir(folder): continue

    files = os.listdir(folder)
    # Stratified: ambil SAMPLE_PER_CLASS secara random per kelas
    sampled = random.sample(files, min(SAMPLE_PER_CLASS, len(files)))

    for file in sampled:
        path = os.path.join(folder, file)
        try:
            feat = extract_features(path)
            X_raw.append(feat)
            y_raw.append(label)
        except Exception as e:
            print(f"Skipped: {file} — {e}")

X_raw = np.array(X_raw)
y_raw = np.array(y_raw)

print(f"Total samples    : {len(X_raw)}")
print(f"Distribusi kelas : {dict(Counter(y_raw))}")

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
print(f"\nClass mapping    : {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# Stratified Split: 70% Train | 15% Val | 15% Test
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
for train_val_idx, test_idx in sss_test.split(X_raw, y_encoded):
    X_train_val, X_test = X_raw[train_val_idx], X_raw[test_idx]
    y_train_val, y_test = y_encoded[train_val_idx], y_encoded[test_idx]

sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.176, random_state=42)
for train_idx, val_idx in sss_val.split(X_train_val, y_train_val):
    X_train, X_val = X_train_val[train_idx], X_train_val[val_idx]
    y_train, y_val = y_train_val[train_idx], y_train_val[val_idx]

print(f"\n{'='*45}")
print(f"  Train  : {len(X_train):>5} samples  ({len(X_train)/len(X_raw)*100:.1f}%)")
print(f"  Val    : {len(X_val):>5} samples  ({len(X_val)/len(X_raw)*100:.1f}%)")
print(f"  Test   : {len(X_test):>5} samples  ({len(X_test)/len(X_raw)*100:.1f}%)")
print(f"{'='*45}")

for split_name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = Counter(y_split)
    total  = len(y_split)
    dist   = {le.classes_[k]: f"{v} ({v/total*100:.1f}%)" for k,v in sorted(counts.items())}
    print(f"{split_name:5} dist: {dist}")


In [ ]:
# PCA Dimensionality Reduction
print("\nApplying PCA...")
pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_val_pca   = pca.transform(X_val)
X_test_pca  = pca.transform(X_test)

print(f"Explained variance (100 components): {np.cumsum(pca.explained_variance_ratio_)[-1]*100:.2f}%")


In [ ]:
# Train & Evaluate Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest"      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "SVM (RBF)"          : SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42, probability=True)
}

results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")

    model.fit(X_train_pca, y_train)

    # Validasi
    y_val_pred = model.predict(X_val_pca)
    val_acc    = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy : {val_acc*100:.2f}%")

    # Final evaluation di test set
    y_test_pred = model.predict(X_test_pca)
    test_acc    = accuracy_score(y_test, y_test_pred)
    print(f"Test Accuracy       : {test_acc*100:.2f}%")
    print("\nClassification Report (Test):")
    print(classification_report(y_test, y_test_pred, target_names=le.classes_))

    results[name] = {
        "model"     : model,
        "val_acc"   : val_acc,
        "test_acc"  : test_acc,
        "y_pred"    : y_test_pred
    }

In [ ]:
best_name  = max(results, key=lambda k: results[k]["val_acc"])
best_model = results[best_name]["model"]
best_pred  = results[best_name]["y_pred"]

print(f"\n✅ Best Model (by Val Acc): {best_name}")
print(f"   Validation Accuracy : {results[best_name]['val_acc']*100:.2f}%")
print(f"   Test Accuracy       : {results[best_name]['test_acc']*100:.2f}%")

# Confusion Matrix
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"Confusion Matrix — {best_name} (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Perbandingan Val vs Test Accuracy

model_names = list(results.keys())
val_accs    = [results[k]["val_acc"] * 100 for k in model_names]
test_accs   = [results[k]["test_acc"] * 100 for k in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, val_accs,  width, label="Validation", color="#4C72B0")
bars2 = ax.bar(x + width/2, test_accs, width, label="Test",       color="#55A868")

ax.set_ylim(0, 105)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Baseline — Val vs Test Accuracy Comparison")
ax.legend()

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


# Model Evaluation (Baseline)


In [ ]:
import matplotlib.patches as mpatches

all_paths, all_labels_raw = [], []
for label in sorted(os.listdir(dataset_path)):
    folder = os.path.join(dataset_path, label)
    if not os.path.isdir(folder): continue
    for file in os.listdir(folder):
        all_paths.append(os.path.join(folder, file))
        all_labels_raw.append(label)

# Fungsi Prediksi Satu Gambar
def predict_single_image(image_path, model, pca_transform, label_encoder):
    """Ekstrak fitur → PCA → prediksi → return label string."""
    feat = extract_features(image_path)
    feat_pca = pca_transform.transform(feat.reshape(1, -1))
    pred_idx = model.predict(feat_pca)[0]
    return label_encoder.inverse_transform([pred_idx])[0]

# Random Sample Testing
N_SAMPLES = 100
sample_indices = random.sample(range(len(all_paths)), N_SAMPLES)
sample_paths   = [all_paths[i] for i in sample_indices]
sample_labels  = [all_labels_raw[i] for i in sample_indices]

print(f"Testing {N_SAMPLES} random samples dengan model: {best_name}\n")

predictions = []
for path in sample_paths:
    pred = predict_single_image(path, best_model, pca, le)
    predictions.append(pred)

# Visualisasi: Gambar + Ground Truth + Prediksi
correct   = sum(p == g for p, g in zip(predictions, sample_labels))
incorrect = N_SAMPLES - correct

fig, axes = plt.subplots(25, 4, figsize=(100, 100))
axes = axes.flatten()

for i, (path, gt, pred) in enumerate(zip(sample_paths, sample_labels, predictions)):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))

    is_correct = (pred == gt)
    border_color = "#2ECC71" if is_correct else "#E74C3C"  # hijau/merah

    axes[i].imshow(img)
    axes[i].set_title(
        f"GT   : {gt}\nPred : {pred}\n{'✅ BENAR' if is_correct else '❌ SALAH'}",
        fontsize=8,
        color="green" if is_correct else "red",
        fontweight="bold"
    )
    for spine in axes[i].spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)

    axes[i].tick_params(left=False, bottom=False,
                        labelleft=False, labelbottom=False)

fig.suptitle(
    f"Random Sample Prediction Check — {best_name}\n"
    f"Benar: {correct}/{N_SAMPLES} ({correct/N_SAMPLES*100:.1f}%)",
    fontsize=14, fontweight="bold", y=1.01
)

green_patch = mpatches.Patch(color="#2ECC71", label=f"Benar ({correct})")
red_patch   = mpatches.Patch(color="#E74C3C", label=f"Salah ({incorrect})")
fig.legend(handles=[green_patch, red_patch], loc="lower center",
           ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.show()

# Tabel Ringkasan Hasil
print(f"\n{'='*60}")
print(f"{'#':>3} | {'Ground Truth':<30} | {'Prediction':<30} | {'Result'}")
print(f"{'='*60}")
for i, (gt, pred) in enumerate(zip(sample_labels, predictions)):
    status = "✅" if pred == gt else "❌"
    print(f"{i+1:>3} | {gt:<30} | {pred:<30} | {status}")
print(f"{'='*60}")
print(f"Accuracy on {N_SAMPLES} random samples: {correct/N_SAMPLES*100:.1f}%")

# Advanced Model

In [ ]:
import os, gc
import shutil
import random

source_dir = "/content/lung_colon_image_set/lung_image_sets"
target_dir = "/tmp/lung_split"

splits = ["train", "val", "test"]
ratios = [0.7, 0.15, 0.15]

# buat folder
for split in splits:
    for cls in os.listdir(source_dir):
        os.makedirs(os.path.join(target_dir, split, cls), exist_ok=True)

# split data
for cls in os.listdir(source_dir):
    files = os.listdir(os.path.join(source_dir, cls))
    random.shuffle(files)

    n = len(files)
    train_end = int(ratios[0] * n)
    val_end = int((ratios[0] + ratios[1]) * n)

    splits_files = {
        "train": files[:train_end],
        "val": files[train_end:val_end],
        "test": files[val_end:]
    }

    for split in splits:
        for file in splits_files[split]:
            src = os.path.join(source_dir, cls, file)
            dst = os.path.join(target_dir, split, cls, file)
            shutil.copy(src, dst)

print("✅ Dataset split selesai")

In [ ]:
# ADVANCED MODEL — MobileNetV2

import numpy as np
import random
import cv2
import matplotlib.patches as mpatches
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (EarlyStopping,
                                        ReduceLROnPlateau,
                                        ModelCheckpoint)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Memory Management
tf.keras.backend.clear_session()
gc.collect()

# Konfigurasi
SPLIT_DIR   = "/tmp/lung_split"
IMG_SIZE    = (160, 160)
BATCH_SIZE  = 16
NUM_CLASSES = 3
AUTOTUNE    = tf.data.AUTOTUNE

for split in ["train", "val", "test"]:
    path = os.path.join(SPLIT_DIR, split)
    assert os.path.exists(path), f"Folder {path} tidak ditemukan! Jalankan cell split dulu."
print("✅ Folder split ditemukan\n")

# tf.data Pipeline
augment_layer = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
], name="augmentation")

def make_dataset(split_name, augment=False):
    directory = os.path.join(SPLIT_DIR, split_name)

    # Load raw
    raw = tf.keras.preprocessing.image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        seed=42,
        shuffle=augment,
        label_mode="int"
    )
    names = raw.class_names

    # Normalize [0,1]
    ds = raw.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y),
                 num_parallel_calls=AUTOTUNE)

    # Augmentasi untuk train
    if augment:
        ds = ds.map(lambda x, y: (augment_layer(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)

    # Prefetch
    ds = ds.prefetch(AUTOTUNE)

    return ds, names

train_ds, class_names = make_dataset("train", augment=True)
val_ds,   _           = make_dataset("val",   augment=False)
test_ds,  _           = make_dataset("test",  augment=False)

print(f"Class names : {class_names}")
print(f"IMG_SIZE    : {IMG_SIZE}")
print(f"Batch size  : {BATCH_SIZE}\n")

# Build MobileNetV2
def build_mobilenetv2(num_classes=NUM_CLASSES, img_size=IMG_SIZE):
    """
    MobileNetV2 — hanya 3.4M parameter (vs ResNet50: 23.5M, EfficientNetB3: 12M)
    Cocok untuk CPU/low-end GPU sekalipun.
    """
    base = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(*img_size, 3),
        alpha=1.0
    )
    base.trainable = False     # Fase 1: frozen

    inputs  = layers.Input(shape=(*img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.3)(x)
    x       = layers.Dense(128, activation="relu")(x)
    x       = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name="MobileNetV2_Lung")

    total_params = model.count_params()
    print(f"Total parameters : {total_params:,}")
    print(f"Trainable params : {sum([tf.size(v).numpy() for v in model.trainable_variables]):,}")

    return model, base

model, base_model = build_mobilenetv2()
model.summary(line_length=80, show_trainable=True)

# Train Head Only
CKPT_PATH = "/tmp/best_mobilenetv2.keras"

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                     patience=2, min_lr=1e-7, verbose=1),
    ModelCheckpoint(CKPT_PATH, monitor="val_accuracy",
                    save_best_only=True, verbose=0)
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("="*55)
print("  FASE 1 — Head Training (base frozen)")
print("="*55)

hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

n_fase1 = len(hist1.history["accuracy"])
print(f"\n✅ Fase 1 selesai — {n_fase1} epoch(s)")
print(f"   Best Val Accuracy: {max(hist1.history['val_accuracy'])*100:.2f}%")

gc.collect()

# Fase 2 — Fine-Tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_after = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"\nFase 2 — Trainable params: {trainable_after:,}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("="*55)
print("  FASE 2 — Fine-Tuning (30 layer terakhir)")
print("="*55)

hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1,
    initial_epoch=n_fase1
)

gc.collect()

# Evaluasi Final di Test Set
print("\n" + "="*55)
print("  EVALUASI TEST SET")
print("="*55)

y_true, y_pred = [], []
for imgs, lbls in test_ds:
    preds = np.argmax(model.predict(imgs, verbose=0), axis=1)
    y_pred.extend(preds)
    y_true.extend(lbls.numpy())

test_acc = accuracy_score(y_true, y_pred)
print(f"\nTest Accuracy: {test_acc*100:.2f}%\n")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix — MobileNetV2 (Test Acc: {test_acc*100:.2f}%)")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png")
plt.show()

# Plot Training History (Fase 1 + Fase 2 gabung)
history = {}

all_keys = set(hist1.history.keys()).union(hist2.history.keys())

for k in all_keys:
    h1 = hist1.history.get(k, [])
    h2 = hist2.history.get(k, [])
    history[k] = h1 + h2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in [(ax1, "accuracy", "Accuracy"), (ax2, "loss", "Loss")]:
    ax.plot(history[metric],             label=f"Train")
    ax.plot(history[f"val_{metric}"],    label=f"Validation")
    ax.axvline(x=n_fase1 - 1, color="gray", linestyle="--",
               alpha=0.7, label="Fine-tune start")
    ax.set_title(f"MobileNetV2 — {title}")
    ax.set_xlabel("Epoch"); ax.set_ylabel(title)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("MobileNetV2 Training History", fontsize=13, fontweight="bold")
plt.tight_layout()

plt.savefig("outputs/training_curve.png")
plt.show()

# Random Sample Prediction Checker
N = 16
sample_batches, sample_labels_true = [], []

for imgs, lbls in test_ds.take(1):
    idxs = random.sample(range(len(imgs)), min(N, len(imgs)))
    sample_imgs  = imgs.numpy()[idxs]
    sample_true  = lbls.numpy()[idxs]

preds_raw = model.predict(
    tf.expand_dims(sample_imgs, 0).numpy().reshape(-1, *IMG_SIZE, 3),
    verbose=0
)
sample_pred = np.argmax(preds_raw, axis=1)
sample_conf = np.max(preds_raw, axis=1)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

correct = 0
for i in range(len(sample_imgs)):
    gt   = class_names[sample_true[i]]
    pred = class_names[sample_pred[i]]
    conf = sample_conf[i] * 100
    ok   = (gt == pred)
    correct += ok

    axes[i].imshow(sample_imgs[i])
    axes[i].set_title(
        f"GT  : {gt}\nPred: {pred}\nConf: {conf:.1f}%\n{'✅' if ok else '❌'}",
        fontsize=8,
        color="green" if ok else "red",
        fontweight="bold"
    )
    for spine in axes[i].spines.values():
        spine.set_edgecolor("#2ECC71" if ok else "#E74C3C")
        spine.set_linewidth(3)
    axes[i].tick_params(left=False, bottom=False,
                        labelleft=False, labelbottom=False)

fig.suptitle(
    f"Prediction Checker — MobileNetV2\n"
    f"Benar: {correct}/{len(sample_imgs)} ({correct/len(sample_imgs)*100:.0f}%)",
    fontsize=14, fontweight="bold"
)
green_p = mpatches.Patch(color="#2ECC71", label=f"Benar ({correct})")
red_p   = mpatches.Patch(color="#E74C3C", label=f"Salah ({len(sample_imgs)-correct})")
fig.legend(handles=[green_p, red_p], loc="lower center",
           ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()

plt.savefig("outputs/prediction_checker.png")
plt.show()

## Final Results

The MobileNetV2 model achieved high accuracy and showed strong generalization performance. Compared to baseline models, deep learning significantly improved classification results.